# Dollar Volume Bars Demo

This notebook demonstrates the dollar volume sampling feature with comprehensive time-based metrics.

## What You'll Learn

1. **Basic Usage** - Creating dollar volume bars from tick data
2. **Time Metrics** - Understanding trading intensity and duration
3. **Market Regimes** - Detecting high/low activity periods
4. **Liquidity Analysis** - Identifying optimal trading windows
5. **Trading Applications** - Generating signals from time metrics

## Why Dollar Volume Bars?

- **Information-driven** sampling (not fixed time intervals)
- **Better statistical properties** (more normal returns)
- **Adaptive to market activity** (more bars when action happens)
- **Time metrics** reveal market microstructure patterns

In [ ]:
# Install if needed (uncomment)
# !pip install -e ..

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Import the dollar volume sampling functions
from binance_tick_data import (
    create_dollar_volume_bars,
    DollarVolumeSampler,
    calculate_optimal_threshold
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Imports successful!")

## 1. Generate Sample Tick Data

Let's create realistic tick data with varying trading patterns.

In [ ]:
def generate_realistic_tick_data(n_ticks=5000, seed=42):
    """
    Generate tick data with realistic patterns:
    - Variable time intervals
    - Session patterns (open/close more active)
    - Random news events (bursts of activity)
    """
    np.random.seed(seed)
    
    # Generate timestamps with realistic patterns
    timestamps = []
    current_time = datetime(2024, 1, 1, 9, 0, 0)  # Market open
    
    for i in range(n_ticks):
        hour = current_time.hour
        
        # Session patterns: faster at open/close, slower at lunch
        if hour in [9, 10, 15, 16]:  # Open/close hours
            avg_interval = 0.5  # Faster trading
        elif hour in [12, 13]:  # Lunch hours
            avg_interval = 3.0  # Slower trading
        else:
            avg_interval = 1.0  # Normal trading
        
        # Random news events (5% chance of burst)
        if np.random.random() < 0.05:
            avg_interval *= 0.1  # 10x faster
        
        interval = np.random.exponential(avg_interval)
        current_time += timedelta(seconds=interval)
        timestamps.append(current_time)
    
    # Generate prices (random walk)
    initial_price = 45000.0
    returns = np.random.normal(0, 0.001, n_ticks)
    prices = initial_price * np.exp(returns.cumsum())
    
    # Generate volumes (log-normal, higher during volatile periods)
    volumes = np.random.lognormal(5, 1.5, n_ticks)
    
    return pd.DataFrame({
        'timestamp': timestamps,
        'price': prices,
        'volume': volumes
    })

# Generate data
tick_data = generate_realistic_tick_data(n_ticks=5000)

print(f"Generated {len(tick_data):,} ticks")
print(f"Time range: {tick_data['timestamp'].min()} to {tick_data['timestamp'].max()}")
print(f"Duration: {(tick_data['timestamp'].max() - tick_data['timestamp'].min()).total_seconds()/3600:.1f} hours")
print(f"\nFirst few rows:")
tick_data.head()

## 2. Create Dollar Volume Bars

Transform tick data into dollar volume bars with automatic threshold calculation.

In [ ]:
# Create dollar volume bars (auto threshold for ~100 ticks per bar)
bars = create_dollar_volume_bars(tick_data, ticks_per_bar=100)

print(f"✅ Created {len(bars)} bars from {len(tick_data):,} ticks")
print(f"\nAverage ticks per bar: {bars['tick_count'].mean():.1f}")
print(f"Average bar duration: {bars['duration_seconds'].mean():.1f} seconds")
print(f"Average tick rate: {bars['ticks_per_second'].mean():.2f} ticks/second")
print(f"\nBar columns: {list(bars.columns)}")
print(f"\nFirst bar:")
bars.head(1).T

## 3. Explore Time Metrics

Let's examine the new time-based metrics that reveal trading patterns.

In [ ]:
# Display time metrics
time_metrics = bars[[
    'bar_id', 'timestamp', 'timestamp_close', 'close',
    'duration_seconds', 'ticks_per_second', 
    'dollar_volume_per_second', 'time_since_last_bar'
]].head(10)

print("First 10 bars - Time Metrics:")
time_metrics

In [ ]:
# Statistical summary of time metrics
print("Time Metrics Statistics:\n")
print(bars[['duration_seconds', 'ticks_per_second', 
           'dollar_volume_per_second', 'time_since_last_bar']].describe())

## 4. Visualize Time Metrics

Visualizations reveal patterns invisible in raw numbers.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Price with duration as color
scatter = axes[0, 0].scatter(range(len(bars)), bars['close'],
                            c=bars['duration_seconds'], 
                            cmap='coolwarm', alpha=0.6, s=30)
axes[0, 0].set_title('Price (colored by bar duration)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Bar Number')
axes[0, 0].set_ylabel('Price ($)')
plt.colorbar(scatter, ax=axes[0, 0], label='Duration (seconds)')

# 2. Trading intensity over time
axes[0, 1].plot(bars['ticks_per_second'], color='green', alpha=0.7, linewidth=1)
axes[0, 1].axhline(bars['ticks_per_second'].mean(), color='red', 
                  linestyle='--', alpha=0.5, 
                  label=f"Mean: {bars['ticks_per_second'].mean():.2f}")
axes[0, 1].fill_between(range(len(bars)), 
                        bars['ticks_per_second'].rolling(5).mean(),
                        alpha=0.3, color='green')
axes[0, 1].set_title('Trading Intensity (Ticks per Second)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Bar Number')
axes[0, 1].set_ylabel('Ticks/Second')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Duration distribution
axes[1, 0].hist(bars['duration_seconds'], bins=30, 
               edgecolor='black', alpha=0.7, color='skyblue')
axes[1, 0].axvline(bars['duration_seconds'].median(), color='red', 
                  linestyle='--', linewidth=2,
                  label=f"Median: {bars['duration_seconds'].median():.1f}s")
axes[1, 0].set_title('Bar Duration Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Duration (seconds)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Dollar volume velocity
axes[1, 1].plot(bars['dollar_volume_per_second'], color='orange', alpha=0.7, linewidth=1)
axes[1, 1].set_title('Dollar Volume Velocity', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Bar Number')
axes[1, 1].set_ylabel('$/Second')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print(f"• Short duration bars correlate with high activity")
print(f"• Tick rate varies significantly ({bars['ticks_per_second'].min():.2f} to {bars['ticks_per_second'].max():.2f})")
print(f"• Dollar velocity shows clear intensity patterns")

## 5. Market Regime Detection

Use time metrics to identify different market regimes.

In [ ]:
# Classify regimes based on trading intensity
intensity_75 = bars['ticks_per_second'].quantile(0.75)
intensity_25 = bars['ticks_per_second'].quantile(0.25)

bars['regime'] = 'normal'
bars.loc[bars['ticks_per_second'] > intensity_75, 'regime'] = 'high_activity'
bars.loc[bars['ticks_per_second'] < intensity_25, 'regime'] = 'low_activity'

# Analyze regime characteristics
regime_stats = bars.groupby('regime').agg({
    'price_change': ['mean', 'std', 'count'],
    'duration_seconds': 'mean',
    'dollar_volume': 'mean',
    'ticks_per_second': 'mean',
    'volume': 'mean'
}).round(2)

print("Market Regime Statistics:")
print(regime_stats)

# Visualize regimes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Price with regime colors
colors = {'high_activity': 'red', 'normal': 'blue', 'low_activity': 'green'}
for regime, color in colors.items():
    mask = bars['regime'] == regime
    ax1.scatter(bars[mask].index, bars[mask]['close'], 
               c=color, label=regime, alpha=0.6, s=20)
ax1.set_title('Price by Market Regime', fontsize=12, fontweight='bold')
ax1.set_xlabel('Bar Number')
ax1.set_ylabel('Price ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Regime distribution
regime_counts = bars['regime'].value_counts()
ax2.bar(regime_counts.index, regime_counts.values, 
       color=[colors[r] for r in regime_counts.index], alpha=0.7)
ax2.set_title('Regime Distribution', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Bars')
for i, (regime, count) in enumerate(regime_counts.items()):
    ax2.text(i, count, f'{count}\n({count/len(bars)*100:.1f}%)', 
            ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\n💡 Insight: High activity periods have {regime_stats.loc['high_activity', ('price_change', 'std')]:.2f} "
      f"vs {regime_stats.loc['low_activity', ('price_change', 'std')]:.2f} price volatility in low activity")

## 6. Liquidity Analysis

Time metrics reveal liquidity patterns critical for execution.

In [ ]:
# Calculate liquidity score
bars['liquidity_score'] = (
    bars['dollar_volume_per_second'] / 
    bars['dollar_volume_per_second'].rolling(20, min_periods=1).mean()
)

# Identify liquidity conditions
low_liquidity = bars[bars['liquidity_score'] < 0.5]
high_liquidity = bars[bars['liquidity_score'] > 1.5]

print("Liquidity Analysis:")
print(f"\nLow liquidity periods: {len(low_liquidity)} bars ({len(low_liquidity)/len(bars)*100:.1f}%)")
print(f"High liquidity periods: {len(high_liquidity)} bars ({len(high_liquidity)/len(bars)*100:.1f}%)")
print(f"\nPrice impact in low liquidity: {abs(low_liquidity['price_change']).mean():.2f}")
print(f"Price impact in high liquidity: {abs(high_liquidity['price_change']).mean():.2f}")

# Trading gaps
gaps = bars['time_since_last_bar'].dropna()
large_gaps = bars[bars['time_since_last_bar'] > gaps.quantile(0.95)]

print(f"\nLarge trading gaps (>95th percentile): {len(large_gaps)}")
print(f"Average large gap: {large_gaps['time_since_last_bar'].mean():.1f} seconds")

# Visualize liquidity
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Liquidity score over time
ax1.plot(bars['liquidity_score'], color='purple', alpha=0.7, linewidth=1)
ax1.axhline(1.0, color='black', linestyle='--', alpha=0.5, label='Average')
ax1.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Low liquidity')
ax1.axhline(1.5, color='green', linestyle='--', alpha=0.5, label='High liquidity')
ax1.fill_between(range(len(bars)), 0.5, 1.5, alpha=0.1, color='gray')
ax1.set_title('Liquidity Score Over Time', fontsize=12, fontweight='bold')
ax1.set_xlabel('Bar Number')
ax1.set_ylabel('Liquidity Score')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Inter-bar gaps
ax2.plot(bars['time_since_last_bar'], color='brown', alpha=0.7, linewidth=1)
ax2.axhline(gaps.median(), color='orange', linestyle='--', 
           label=f'Median: {gaps.median():.1f}s')
ax2.set_title('Time Between Bars (Trading Gaps)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Bar Number')
ax2.set_ylabel('Gap (seconds)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Trading Signal Generation

Use time metrics to generate entry/exit signals.

In [ ]:
# Calculate technical indicators with time metrics
bars['intensity_ma'] = bars['ticks_per_second'].rolling(20).mean()
bars['velocity_ma'] = bars['dollar_volume_per_second'].rolling(20).mean()
bars['price_ma'] = bars['close'].rolling(20).mean()

# Generate signals
bars['signal'] = 0

# Entry signal: High intensity + high velocity + positive price momentum
entry_condition = (
    (bars['ticks_per_second'] > bars['intensity_ma']) &
    (bars['dollar_volume_per_second'] > bars['velocity_ma']) &
    (bars['close'] > bars['price_ma']) &
    (bars['price_change'] > 0)
)
bars.loc[entry_condition, 'signal'] = 1

# Exit signal: Declining intensity
exit_condition = (
    (bars['ticks_per_second'] < bars['intensity_ma'] * 0.7)
)
bars.loc[exit_condition, 'signal'] = -1

# Count signals
entry_signals = (bars['signal'] == 1).sum()
exit_signals = (bars['signal'] == -1).sum()

print(f"\n📈 Trading Signals Generated:")
print(f"Entry signals: {entry_signals} ({entry_signals/len(bars)*100:.1f}%)")
print(f"Exit signals: {exit_signals} ({exit_signals/len(bars)*100:.1f}%)")

# Visualize signals
fig, ax = plt.subplots(figsize=(14, 6))

# Plot price
ax.plot(bars.index, bars['close'], color='blue', alpha=0.7, linewidth=1, label='Price')
ax.plot(bars.index, bars['price_ma'], color='gray', alpha=0.5, 
       linestyle='--', label='20-bar MA')

# Mark entry signals
entry_bars = bars[bars['signal'] == 1]
ax.scatter(entry_bars.index, entry_bars['close'], 
          color='green', marker='^', s=100, label='Entry', zorder=5)

# Mark exit signals
exit_bars = bars[bars['signal'] == -1]
ax.scatter(exit_bars.index, exit_bars['close'], 
          color='red', marker='v', s=100, label='Exit', zorder=5)

ax.set_title('Trading Signals Based on Time Metrics', fontsize=12, fontweight='bold')
ax.set_xlabel('Bar Number')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Signal quality analysis
if entry_signals > 0:
    entry_bars_data = bars[bars['signal'] == 1]
    avg_intensity = entry_bars_data['ticks_per_second'].mean()
    avg_change = entry_bars_data['price_change'].mean()
    print(f"\n💡 Entry Signal Characteristics:")
    print(f"  Average intensity at entry: {avg_intensity:.2f} ticks/sec")
    print(f"  Average price change on entry bar: ${avg_change:.2f}")

## 8. Comparison: Dollar Bars vs Time Bars

Compare dollar volume bars with traditional time-based bars.

In [ ]:
# Create time-based bars (1-minute intervals)
tick_data_indexed = tick_data.set_index('timestamp')
time_bars = tick_data_indexed.resample('1min').agg({
    'price': ['first', 'max', 'min', 'last'],
    'volume': 'sum'
}).dropna()

time_bars.columns = ['open', 'high', 'low', 'close', 'volume']
time_bars = time_bars[time_bars['volume'] > 0]

# Calculate returns
bars['returns'] = bars['close'].pct_change()
time_bars['returns'] = time_bars['close'].pct_change()

# Compare distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Returns distribution - Dollar bars
axes[0, 0].hist(bars['returns'].dropna(), bins=30, 
               edgecolor='black', alpha=0.7, color='green')
axes[0, 0].axvline(0, color='red', linestyle='--')
axes[0, 0].set_title('Dollar Bars - Returns Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Returns')
axes[0, 0].set_ylabel('Frequency')

# Returns distribution - Time bars
axes[0, 1].hist(time_bars['returns'].dropna(), bins=30, 
               edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].axvline(0, color='red', linestyle='--')
axes[0, 1].set_title('Time Bars (1min) - Returns Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Returns')
axes[0, 1].set_ylabel('Frequency')

# Q-Q plot - Dollar bars
from scipy import stats
stats.probplot(bars['returns'].dropna(), dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Dollar Bars - Q-Q Plot', fontweight='bold')

# Q-Q plot - Time bars
stats.probplot(time_bars['returns'].dropna(), dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Time Bars - Q-Q Plot', fontweight='bold')

plt.tight_layout()
plt.show()

# Statistical comparison
print("\nStatistical Comparison:")
print("\nDollar Bars:")
print(f"  Mean return: {bars['returns'].mean():.6f}")
print(f"  Std deviation: {bars['returns'].std():.6f}")
print(f"  Skewness: {bars['returns'].skew():.3f}")
print(f"  Kurtosis: {bars['returns'].kurtosis():.3f}")

print("\nTime Bars:")
print(f"  Mean return: {time_bars['returns'].mean():.6f}")
print(f"  Std deviation: {time_bars['returns'].std():.6f}")
print(f"  Skewness: {time_bars['returns'].skew():.3f}")
print(f"  Kurtosis: {time_bars['returns'].kurtosis():.3f}")

# Normality test
_, dollar_p = stats.jarque_bera(bars['returns'].dropna())
_, time_p = stats.jarque_bera(time_bars['returns'].dropna())

print(f"\nJarque-Bera Normality Test:")
print(f"  Dollar bars p-value: {dollar_p:.4f}")
print(f"  Time bars p-value: {time_p:.4f}")
print(f"\n✅ Dollar bars {'ARE' if dollar_p > time_p else 'are NOT'} more normally distributed")

## 9. Advanced: Adaptive Threshold

Use adaptive thresholds that adjust to market activity.

In [ ]:
# Create adaptive bars
adaptive_bars = create_dollar_volume_bars(
    tick_data,
    ticks_per_bar=100,
    adaptive=True,
    lookback_bars=20
)

print(f"Created {len(adaptive_bars)} adaptive bars")
print(f"\nThreshold variation:")
print(f"  Min: ${adaptive_bars['threshold_used'].min():,.0f}")
print(f"  Max: ${adaptive_bars['threshold_used'].max():,.0f}")
print(f"  Mean: ${adaptive_bars['threshold_used'].mean():,.0f}")
print(f"  Std: ${adaptive_bars['threshold_used'].std():,.0f}")

# Visualize threshold adaptation
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Threshold over time
ax1.plot(adaptive_bars['threshold_used'], color='blue', alpha=0.7, linewidth=1.5)
ax1.axhline(adaptive_bars['threshold_used'].mean(), color='red', 
           linestyle='--', alpha=0.5, label='Mean')
ax1.fill_between(range(len(adaptive_bars)),
                adaptive_bars['threshold_used'].rolling(10).mean(),
                alpha=0.3, color='blue')
ax1.set_title('Adaptive Threshold Over Time', fontsize=12, fontweight='bold')
ax1.set_xlabel('Bar Number')
ax1.set_ylabel('Dollar Volume Threshold ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Tick count stability
ax2.plot(adaptive_bars['tick_count'], color='green', alpha=0.7, linewidth=1, label='Adaptive')
ax2.plot(bars['tick_count'], color='orange', alpha=0.5, linewidth=1, label='Fixed')
ax2.axhline(100, color='red', linestyle='--', alpha=0.5, label='Target')
ax2.set_title('Tick Count per Bar: Adaptive vs Fixed', fontsize=12, fontweight='bold')
ax2.set_xlabel('Bar Number')
ax2.set_ylabel('Ticks per Bar')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Tick count stability:")
print(f"  Adaptive bars: {adaptive_bars['tick_count'].std():.1f} std deviation")
print(f"  Fixed bars: {bars['tick_count'].std():.1f} std deviation")
print(f"  ✅ Adaptive is {bars['tick_count'].std() / adaptive_bars['tick_count'].std():.1f}x more stable")

## 10. Summary and Key Takeaways

What we learned about dollar volume bars with time metrics.

In [ ]:
print("="*80)
print("DOLLAR VOLUME BARS - KEY TAKEAWAYS")
print("="*80)

print("\n✅ CORE BENEFITS:")
print("  1. Information-driven sampling adapts to market activity")
print("  2. Better statistical properties than time-based bars")
print("  3. Time metrics reveal market microstructure patterns")
print("  4. Enables regime detection and liquidity analysis")

print("\n📊 TIME METRICS INSIGHTS:")
print(f"  • ticks_per_second: Trading intensity measurement")
print(f"    Range: {bars['ticks_per_second'].min():.2f} - {bars['ticks_per_second'].max():.2f}")
print(f"  • duration_seconds: Bar time span (inversely correlates with activity)")
print(f"    Average: {bars['duration_seconds'].mean():.1f}s")
print(f"  • dollar_volume_per_second: Capital flow velocity")
print(f"    Average: ${bars['dollar_volume_per_second'].mean():,.0f}/sec")
print(f"  • time_since_last_bar: Detects trading gaps and regime changes")
print(f"    Max gap: {bars['time_since_last_bar'].max():.1f}s")

print("\n💡 TRADING APPLICATIONS:")
print("  • Entry timing: High intensity + high velocity = strong momentum")
print("  • Exit timing: Declining intensity = weakening interest")
print("  • Risk management: Adjust position sizes based on regime")
print("  • Execution: Trade during high liquidity windows")
print("  • News detection: Intensity spikes reveal events")

print("\n🎯 BEST PRACTICES:")
print("  1. Start with 100 ticks per bar, adjust based on strategy")
print("  2. Use adaptive mode for multi-day datasets")
print("  3. Combine time metrics with traditional indicators")
print("  4. Monitor regime changes for strategy adaptation")
print("  5. Backtest across different market conditions")

print("\n" + "="*80)
print("Ready to apply dollar volume bars to your trading!")
print("="*80)

## Next Steps

1. **Try with real data**: Load your own Binance tick data using `BinanceDataRepository`
2. **Experiment with thresholds**: Test different `ticks_per_bar` values
3. **Build strategies**: Use time metrics in your trading logic
4. **Backtest**: Validate performance across different periods
5. **Combine with other features**: Use with technical indicators, ML models, etc.

## Documentation

- Full docs: `docs/DOLLAR_VOLUME_SAMPLING.md`
- Time metrics guide: `TIME_METRICS_ENHANCEMENT.md`
- Examples: `examples/dollar_volume_bars_example.py`
- Tests: `tests/test_dollar_volume_sampling.py`